In [1]:
import subprocess
import sys

libraries = [
    "numpy",
    "tensorflow",
    "scikit-learn",
    "matplotlib",
    "seaborn",
    "pillow"
]

print("Installing required libraries...\n")

for lib in libraries:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", lib])
        print(f"{lib} installed successfully!")
    except Exception as e:
        print(f"Error installing {lib}: {e}")

print("\nAll required libraries installed successfully!")

import numpy as np
import os
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import json
import matplotlib.pyplot as plt
import seaborn as sns

Installing required libraries...

numpy installed successfully!
tensorflow installed successfully!
scikit-learn installed successfully!
matplotlib installed successfully!
seaborn installed successfully!
pillow installed successfully!

All required libraries installed successfully!


In [2]:
# Define image dimensions and batch size
img_height, img_width = 128, 128
batch_size = 32

# Define directories
test_data_dir = "../dataset/test"
model_dir = "../model_saved_files"

In [3]:
# Test data preprocessing (only rescale, no augmentation)
test_datagen = ImageDataGenerator(rescale=1./255)

In [4]:
# Load test data
test_generator = test_datagen.flow_from_directory(
    test_data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False  # Important: Don't shuffle for consistent evaluation
)

print(f"✓ Test set loaded")
print(f"  Total samples: {test_generator.samples}")
print(f"  Number of classes: {test_generator.num_classes}")
print(f"  Classes: {list(test_generator.class_indices.keys())}")

Found 428 images belonging to 14 classes.
✓ Test set loaded
  Total samples: 428
  Number of classes: 14
  Classes: ['Dariers disease', 'Lindsays nails', 'aloperia areata', 'beaus lines', 'bluish nail', 'bulging eyes', 'cataracts eyes', 'clubbing', 'crossed eyes', 'eczema', 'glucoma eyes', 'lip', 'tounge', 'uvieties eyes']


In [5]:
# Define all models to load (based on your actual .h5 files)
model_files = {
    'CNN': 'Cnn.h5',
    'EfficientNetV2L': 'EfficientNetV2L.h5',
    'InceptionResNetV2': 'InceptionResNetV2.h5',
    'InceptionV3': 'InceptionV3.h5',
    'MobileNet': 'Mobilenet.h5',
    'ResNet': 'ResNet.h5',
    'VGG16': 'VGG16.h5',
    'Xception': 'Xception.h5'
}

# Load all models
models = {}
available_models = []

print("Loading models...")
print("=" * 60)

for model_name, model_file in model_files.items():
    model_path = os.path.join(model_dir, model_file)
    if os.path.exists(model_path):
        try:
            models[model_name] = load_model(model_path)
            available_models.append(model_name)
            print(f"✓ Loaded {model_name:20s} from {model_file}")
        except Exception as e:
            print(f"✗ Failed to load {model_name}: {e}")
    else:
        print(f"⚠ {model_name} not found at {model_path}")

print("=" * 60)
print(f"\n✓ Total models loaded: {len(models)}")

Loading models...


✓ Loaded CNN                  from Cnn.h5


✓ Loaded EfficientNetV2L      from EfficientNetV2L.h5
✗ Failed to load InceptionResNetV2: Unknown layer: 'CustomScaleLayer'. Please ensure you are using a `keras.utils.custom_object_scope` and that this object is included in the scope. See https://www.tensorflow.org/guide/keras/save_and_serialize#registering_the_custom_object for details.


✓ Loaded InceptionV3          from InceptionV3.h5


✓ Loaded MobileNet            from Mobilenet.h5


✓ Loaded ResNet               from ResNet.h5
✓ Loaded VGG16                from VGG16.h5


✓ Loaded Xception             from Xception.h5

✓ Total models loaded: 7


In [6]:
# Get true labels from test generator
test_generator.reset()
y_true = test_generator.classes

print(f"True labels shape: {y_true.shape}")
print(f"Unique classes: {np.unique(y_true)}")
print(f"Class distribution:")
unique, counts = np.unique(y_true, return_counts=True)
for cls, count in zip(unique, counts):
    class_name = list(test_generator.class_indices.keys())[cls]
    print(f"  Class {cls} ({class_name}): {count} samples")

True labels shape: (428,)
Unique classes: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13]
Class distribution:
  Class 0 (Dariers disease): 17 samples
  Class 1 (Lindsays nails): 18 samples
  Class 2 (aloperia areata): 24 samples
  Class 3 (beaus lines): 103 samples
  Class 4 (bluish nail): 19 samples
  Class 5 (bulging eyes): 18 samples
  Class 6 (cataracts eyes): 15 samples
  Class 7 (clubbing): 23 samples
  Class 8 (crossed eyes): 16 samples
  Class 9 (eczema): 18 samples
  Class 10 (glucoma eyes): 104 samples
  Class 11 (lip): 17 samples
  Class 12 (tounge): 18 samples
  Class 13 (uvieties eyes): 18 samples


In [7]:
# Get predictions from each model
all_predictions = {}
model_accuracies = {}

print("\nGetting predictions from each model...")
print("=" * 80)

for model_name in available_models:
    print(f"\nProcessing {model_name}...")
    test_generator.reset()
    predictions = models[model_name].predict(test_generator, verbose=0)
    all_predictions[model_name] = predictions
    
    # Calculate individual model accuracy
    y_pred = np.argmax(predictions, axis=1)
    accuracy = accuracy_score(y_true, y_pred)
    model_accuracies[model_name] = accuracy
    print(f"  {model_name:20s} Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

print("=" * 80)
print("\n✓ All predictions collected")


Getting predictions from each model...

Processing CNN...
  CNN                  Accuracy: 0.6379 (63.79%)

Processing EfficientNetV2L...
  EfficientNetV2L      Accuracy: 0.2664 (26.64%)

Processing InceptionV3...
  InceptionV3          Accuracy: 0.7477 (74.77%)

Processing MobileNet...
  MobileNet            Accuracy: 0.8248 (82.48%)

Processing ResNet...
  ResNet               Accuracy: 0.1332 (13.32%)

Processing VGG16...
  VGG16                Accuracy: 0.6005 (60.05%)

Processing Xception...
  Xception             Accuracy: 0.7336 (73.36%)

✓ All predictions collected


In [8]:
# ENSEMBLE METHOD 1: SOFT VOTING (Average Probabilities)
print("\n" + "=" * 80)
print("ENSEMBLE METHOD 1: SOFT VOTING (Average Probabilities)")
print("=" * 80)

# Stack all predictions and average them
stacked_predictions = np.array([all_predictions[model] for model in available_models])
soft_ensemble_predictions = np.mean(stacked_predictions, axis=0)
y_pred_soft = np.argmax(soft_ensemble_predictions, axis=1)

soft_accuracy = accuracy_score(y_true, y_pred_soft)
print(f"\n✓ Soft Voting Ensemble Accuracy: {soft_accuracy:.4f} ({soft_accuracy*100:.2f}%)")

# Show improvement
best_individual = max(model_accuracies.values())
improvement = soft_accuracy - best_individual
print(f"  Improvement over best single model: +{improvement:.4f} (+{improvement*100:.2f}%)")


ENSEMBLE METHOD 1: SOFT VOTING (Average Probabilities)

✓ Soft Voting Ensemble Accuracy: 0.5444 (54.44%)
  Improvement over best single model: +-0.2804 (+-28.04%)


In [9]:
# ENSEMBLE METHOD 2: HARD VOTING (Majority Vote)
print("\n" + "=" * 80)
print("ENSEMBLE METHOD 2: HARD VOTING (Majority Vote)")
print("=" * 80)

# Convert all predictions to class labels
all_class_preds = []
for model in available_models:
    class_preds = np.argmax(all_predictions[model], axis=1)
    all_class_preds.append(class_preds)

all_class_preds_array = np.array(all_class_preds)

# Hard voting: select most common class
hard_ensemble_predictions = []
for i in range(len(y_true)):
    votes = all_class_preds_array[:, i]
    # Find most common prediction
    most_common = np.bincount(votes).argmax()
    hard_ensemble_predictions.append(most_common)

y_pred_hard = np.array(hard_ensemble_predictions)
hard_accuracy = accuracy_score(y_true, y_pred_hard)
print(f"\n✓ Hard Voting Ensemble Accuracy: {hard_accuracy:.4f} ({hard_accuracy*100:.2f}%)")

improvement = hard_accuracy - best_individual
print(f"  Improvement over best single model: +{improvement:.4f} (+{improvement*100:.2f}%)")


ENSEMBLE METHOD 2: HARD VOTING (Majority Vote)

✓ Hard Voting Ensemble Accuracy: 0.7734 (77.34%)
  Improvement over best single model: +-0.0514 (+-5.14%)


In [10]:
# ENSEMBLE METHOD 3: WEIGHTED VOTING (Based on Individual Model Accuracy)
print("\n" + "=" * 80)
print("ENSEMBLE METHOD 3: WEIGHTED VOTING")
print("=" * 80)

# Normalize model accuracies as weights
total_accuracy = sum(model_accuracies.values())
weights = {model: acc / total_accuracy for model, acc in model_accuracies.items()}

print("\nModel Weights:")
for model, weight in weights.items():
    print(f"  {model:20s}: {weight:.4f} (acc={model_accuracies[model]:.4f})")

# Weighted soft voting
weighted_predictions = np.zeros_like(all_predictions[available_models[0]])
for model in available_models:
    weighted_predictions += weights[model] * all_predictions[model]

y_pred_weighted = np.argmax(weighted_predictions, axis=1)
weighted_accuracy = accuracy_score(y_true, y_pred_weighted)

print(f"\n✓ Weighted Voting Ensemble Accuracy: {weighted_accuracy:.4f} ({weighted_accuracy*100:.2f}%)")
improvement = weighted_accuracy - best_individual
print(f"  Improvement over best single model: +{improvement:.4f} (+{improvement*100:.2f}%)")


ENSEMBLE METHOD 3: WEIGHTED VOTING

Model Weights:
  CNN                 : 0.1617 (acc=0.6379)
  EfficientNetV2L     : 0.0675 (acc=0.2664)
  InceptionV3         : 0.1896 (acc=0.7477)
  MobileNet           : 0.2091 (acc=0.8248)
  ResNet              : 0.0338 (acc=0.1332)
  VGG16               : 0.1523 (acc=0.6005)
  Xception            : 0.1860 (acc=0.7336)

✓ Weighted Voting Ensemble Accuracy: 0.7313 (73.13%)
  Improvement over best single model: +-0.0935 (+-9.35%)


In [11]:
# FINAL COMPARISON SUMMARY
print("\n" + "=" * 80)
print("ACCURACY COMPARISON SUMMARY")
print("=" * 80)

results = {
    "Soft Voting Ensemble": soft_accuracy,
    "Hard Voting Ensemble": hard_accuracy,
    "Weighted Voting Ensemble": weighted_accuracy,
}
results.update(model_accuracies)

sorted_results = sorted(results.items(), key=lambda x: x[1], reverse=True)

print(f"\n{'Model':<30} {'Accuracy':<12}")
print("-" * 46)
for name, acc in sorted_results:
    label = " <- BEST" if acc == sorted_results[0][1] else ""
    print(f"{name:<30} {acc:.4f}{label}")

best_individual_acc = max(model_accuracies.values())
best_ensemble_acc = max(soft_accuracy, hard_accuracy, weighted_accuracy)
ensemble_gain = best_ensemble_acc - best_individual_acc

print("\n" + "=" * 80)
print(f"Best Individual Accuracy : {best_individual_acc:.4f}")
print(f"Best Ensemble Accuracy   : {best_ensemble_acc:.4f}")
print(f"Ensemble Gain            : +{ensemble_gain:.4f} (+{ensemble_gain*100:.2f}%)")
print("=" * 80)


ACCURACY COMPARISON SUMMARY

Model                          Accuracy    
----------------------------------------------
MobileNet                      0.8248 <- BEST
Hard Voting Ensemble           0.7734
InceptionV3                    0.7477
Xception                       0.7336
Weighted Voting Ensemble       0.7313
CNN                            0.6379
VGG16                          0.6005
Soft Voting Ensemble           0.5444
EfficientNetV2L                0.2664
ResNet                         0.1332

Best Individual Accuracy : 0.8248
Best Ensemble Accuracy   : 0.7734
Ensemble Gain            : +-0.0514 (+-5.14%)


In [12]:
# SAVE ENSEMBLE METADATA FOR FLASK APP
best_method = max(
    {
        "soft_voting": soft_accuracy,
        "hard_voting": hard_accuracy,
        "weighted_voting": weighted_accuracy,
    },
    key=lambda k: {
        "soft_voting": soft_accuracy,
        "hard_voting": hard_accuracy,
        "weighted_voting": weighted_accuracy,
    }[k],
)

ensemble_metadata = {
    "best_method": best_method,
    "soft_voting_accuracy": float(soft_accuracy),
    "hard_voting_accuracy": float(hard_accuracy),
    "weighted_voting_accuracy": float(weighted_accuracy),
    "best_individual_accuracy": float(best_individual_acc),
    "best_ensemble_accuracy": float(best_ensemble_acc),
    "ensemble_gain": float(ensemble_gain),
    "models_used": available_models,
    "model_accuracies": {model: float(acc) for model, acc in model_accuracies.items()},
    "model_weights": {model: float(w) for model, w in weights.items()},
    "class_names": list(test_generator.class_indices.keys()),
    "num_classes": int(test_generator.num_classes),
}

metadata_path = os.path.join(model_dir, "ensemble_metadata.json")
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(ensemble_metadata, f, indent=2)

print(f"\n✓ Saved ensemble metadata to: {metadata_path}")
print(json.dumps(ensemble_metadata, indent=2))


✓ Saved ensemble metadata to: ../model_saved_files\ensemble_metadata.json
{
  "best_method": "hard_voting",
  "soft_voting_accuracy": 0.544392523364486,
  "hard_voting_accuracy": 0.7733644859813084,
  "weighted_voting_accuracy": 0.7313084112149533,
  "best_individual_accuracy": 0.8247663551401869,
  "best_ensemble_accuracy": 0.7733644859813084,
  "ensemble_gain": -0.051401869158878566,
  "models_used": [
    "CNN",
    "EfficientNetV2L",
    "InceptionV3",
    "MobileNet",
    "ResNet",
    "VGG16",
    "Xception"
  ],
  "model_accuracies": {
    "CNN": 0.6378504672897196,
    "EfficientNetV2L": 0.26635514018691586,
    "InceptionV3": 0.7476635514018691,
    "MobileNet": 0.8247663551401869,
    "ResNet": 0.13317757009345793,
    "VGG16": 0.6004672897196262,
    "Xception": 0.7336448598130841
  },
  "model_weights": {
    "CNN": 0.16172985781990523,
    "EfficientNetV2L": 0.06753554502369669,
    "InceptionV3": 0.18957345971563982,
    "MobileNet": 0.20912322274881517,
    "ResNet": 0.